# Day 08 · Trees: decisions a human can read

**Goal:** grow a decision tree, **read** it as if/else rules, understand **Gini / entropy / information gain** by computing them by hand, watch a tree **overfit**, and fix it with **pre-pruning** (depth, leaf size) and **post-pruning** (`ccp_alpha` via cost-complexity pruning). Finish with feature importance done honestly.

**Run in Google Colab:** File → Open notebook → Upload → pick this `.ipynb`.

**Runtime:** CPU. Every tree here trains in milliseconds.

| Section | What you'll do |
|---|---|
| 1 | Titanic: a dataset with readable questions |
| 2 | Gini, entropy and information gain by hand |
| 3 | Grow a small tree, read it, draw it |
| 4 | Overfit on purpose: depth sweep and a 2-D picture |
| 5 | Post-pruning with cost-complexity (`ccp_alpha`) |
| 6 | Feature importance: impurity-based vs permutation |
| 7 | 🎯 Try it yourself |
| 8 | Homework starter (Kaggle submission) + key takeaways |

## 0. Setup

Everything is preinstalled. `dtreeviz` (prettier tree plots) is optional and commented out.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, sklearn
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.metrics import accuracy_score

SEED = 42
plt.rcParams["figure.figsize"] = (8, 4.5)
print("numpy", np.__version__, "| pandas", pd.__version__, "| sklearn", sklearn.__version__)
# Optional, for beautiful tree diagrams (takes ~30 s to install):
# !pip install -q dtreeviz

## 1. Titanic: a dataset with readable questions 🚢

**What:** 891 passengers; we keep `pclass`, `sex`, `age`, `sibsp` (siblings/spouses aboard), `parch` (parents/children aboard), `fare`, and predict `survived`.

**Why this dataset for trees:** the rules the tree discovers ("women first", "third-class men mostly died") are historically true and easy to read, so you can check whether the tree is *sensible*, not just accurate. Note what we do **not** do: no scaling, no one-hot for a binary column. Trees do not care.

In [ ]:
URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
raw = pd.read_csv(URL)
df = raw[["survived", "pclass", "sex", "age", "sibsp", "parch", "fare"]].dropna().copy()
df["sex"] = (df["sex"] == "male").astype(int)           # 1 = male, 0 = female; a threshold at 0.5 means "is male?"

X, y = df.drop(columns="survived"), df["survived"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=SEED, stratify=y)
print(df.shape, "| survival rate:", y.mean().round(3))
df.head()

## 2. Gini, entropy and information gain by hand ✍️

$$\text{Gini}(S) = 1 - \sum_k p_k^2 \qquad \text{Entropy}(S) = -\sum_k p_k \log_2 p_k \qquad \text{Gain} = I(S) - \sum_c \frac{|S_c|}{|S|}\, I(S_c)$$

We compute the root node's impurity, then the gain of the two most obvious candidate questions, **"is male?"** and **"first class?"**, and see which one the tree *should* pick first. Then we let sklearn confirm.

In [ ]:
def gini(labels):
    p = np.bincount(labels, minlength=2) / len(labels)
    return 1 - (p ** 2).sum()

def entropy(labels):
    p = np.bincount(labels, minlength=2) / len(labels)
    p = p[p > 0]
    return -(p * np.log2(p)).sum()

def gain(labels, mask, impurity=gini):
    left, right = labels[mask], labels[~mask]
    w_child = (len(left) * impurity(left) + len(right) * impurity(right)) / len(labels)
    return impurity(labels) - w_child

yt = y_tr.values
print(f"root: {len(yt)} passengers, gini={gini(yt):.3f}, entropy={entropy(yt):.3f} bits")
for name, mask in [("is male?", X_tr.sex.values == 1), ("first class?", X_tr.pclass.values == 1),
                   ("age < 10?", X_tr.age.values < 10), ("fare > 50?", X_tr.fare.values > 50)]:
    print(f"  split '{name:13s}': gini gain={gain(yt, mask):.3f}   entropy gain={gain(yt, mask, entropy):.3f}")
print("\nThe tree should ask 'is male?' first. Let's check.")

## 3. Grow a small tree, read it, draw it 🌱

`max_depth=3` keeps it readable. `export_text` prints the rules; `plot_tree` draws the boxes. Read the first split: is it what Section 2 predicted?

In [ ]:
tree3 = DecisionTreeClassifier(max_depth=3, random_state=SEED).fit(X_tr, y_tr)
print(export_text(tree3, feature_names=list(X.columns), show_weights=True))
print(f"train accuracy {tree3.score(X_tr, y_tr):.3f} | test accuracy {tree3.score(X_te, y_te):.3f}")

In [ ]:
plt.figure(figsize=(20, 8))
plot_tree(tree3, feature_names=list(X.columns), class_names=["died", "survived"],
          filled=True, rounded=True, proportion=True, fontsize=9)
plt.title("Depth-3 Titanic tree. Read each box: question / gini / share of samples / class split / majority")
plt.show()
# Optional pretty version:
# import dtreeviz
# viz = dtreeviz.model(tree3, X_tr, y_tr, feature_names=list(X.columns), target_name="survived", class_names=["died", "survived"])
# viz.view()

In [ ]:
# Follow ONE passenger through the tree: a 25-year-old woman in 2nd class, fare 30
passenger = pd.DataFrame([{"pclass": 2, "sex": 0, "age": 25, "sibsp": 0, "parch": 0, "fare": 30}])
path = tree3.decision_path(passenger).indices
feat, thr = tree3.tree_.feature, tree3.tree_.threshold
for node in path:
    if feat[node] >= 0:
        f = X.columns[feat[node]]; v = passenger.iloc[0][f]
        print(f"node {node}: {f} = {v} {'<=' if v <= thr[node] else '>'} {thr[node]:.2f}")
    else:
        print(f"leaf {node}: predict {'survived' if tree3.predict(passenger)[0] else 'died'} with P(survived)={tree3.predict_proba(passenger)[0,1]:.2f}")
print("\nThis path IS the explanation you give the customer / regulator.")

## 4. Overfit on purpose 📈

With `max_depth=None` (the default) the tree keeps splitting until every leaf is pure. On real data that means memorising individual passengers. Watch the train/test gap open as depth grows, then see the same thing as a picture on a 2-D toy dataset.

In [ ]:
full = DecisionTreeClassifier(random_state=SEED).fit(X_tr, y_tr)
print(f"unrestricted tree: depth={full.get_depth()}  leaves={full.get_n_leaves()}  train={full.score(X_tr, y_tr):.3f}  test={full.score(X_te, y_te):.3f}")

rows = []
for d in [1, 2, 3, 4, 5, 6, 8, 10, 15, None]:
    t = DecisionTreeClassifier(max_depth=d, random_state=SEED).fit(X_tr, y_tr)
    rows.append({"max_depth": str(d), "leaves": t.get_n_leaves(), "train": t.score(X_tr, y_tr), "test": t.score(X_te, y_te),
                 "cv5": cross_val_score(DecisionTreeClassifier(max_depth=d, random_state=SEED), X_tr, y_tr, cv=5).mean()})
sweep = pd.DataFrame(rows).set_index("max_depth")
print(sweep.round(3))
sweep[["train", "test", "cv5"]].plot(marker="o"); plt.ylabel("accuracy"); plt.title("Train keeps rising; test/CV peak early then fall: overfitting"); plt.show()

In [ ]:
# The same story as a picture: decision regions on a 2-D dataset at three depths
from sklearn.datasets import make_moons
from sklearn.inspection import DecisionBoundaryDisplay
Xm, ym = make_moons(n_samples=300, noise=0.3, random_state=SEED)
Xm_tr, Xm_te, ym_tr, ym_te = train_test_split(Xm, ym, test_size=0.3, random_state=SEED)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.3))
for ax, d in zip(axes, [2, 5, None]):
    m = DecisionTreeClassifier(max_depth=d, random_state=SEED).fit(Xm_tr, ym_tr)
    DecisionBoundaryDisplay.from_estimator(m, Xm, ax=ax, alpha=.35, cmap="RdYlGn", response_method="predict")
    ax.scatter(Xm_tr[:, 0], Xm_tr[:, 1], c=ym_tr, cmap="RdYlGn", edgecolor="k", s=18)
    ax.set_title(f"max_depth={d}: leaves={m.get_n_leaves()}, train={m.score(Xm_tr, ym_tr):.2f}, test={m.score(Xm_te, ym_te):.2f}")
plt.suptitle("Axis-aligned rectangles. Unlimited depth carves a rectangle around every noisy point."); plt.tight_layout(); plt.show()

## 5. Post-pruning with cost-complexity pruning ✂️

$$R_\alpha(T) = R(T) + \alpha\,|\text{leaves}(T)|$$

Grow the full tree, then charge **α per leaf**. `cost_complexity_pruning_path` returns every α at which some branch stops being worth its leaves. We cross-validate over those α values and keep the winner. One knob, chosen by data, instead of guessing a depth.

In [ ]:
path = full.cost_complexity_pruning_path(X_tr, y_tr)
alphas = path.ccp_alphas[:-1]                      # the last alpha prunes everything to a single leaf; skip it
print(f"{len(alphas)} candidate alphas, from {alphas.min():.5f} to {alphas.max():.4f}")

cv_scores, n_leaves = [], []
for a in alphas:
    t = DecisionTreeClassifier(ccp_alpha=a, random_state=SEED)
    cv_scores.append(cross_val_score(t, X_tr, y_tr, cv=5).mean())
    n_leaves.append(t.fit(X_tr, y_tr).get_n_leaves())

best_alpha = alphas[int(np.argmax(cv_scores))]
pruned = DecisionTreeClassifier(ccp_alpha=best_alpha, random_state=SEED).fit(X_tr, y_tr)
print(f"best alpha={best_alpha:.4f}: leaves {full.get_n_leaves()} -> {pruned.get_n_leaves()}, depth {full.get_depth()} -> {pruned.get_depth()}")
print(f"test accuracy: full {full.score(X_te, y_te):.3f} -> pruned {pruned.score(X_te, y_te):.3f}")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(alphas, n_leaves, marker=".", c="#4FD3F2"); ax[0].set_xscale("log"); ax[0].set_xlabel("ccp_alpha"); ax[0].set_ylabel("leaves"); ax[0].set_title("Higher price per leaf → fewer leaves")
ax[1].plot(alphas, cv_scores, marker=".", c="#FFB43A"); ax[1].set_xscale("log"); ax[1].axvline(best_alpha, ls="--", c="k", lw=1); ax[1].set_xlabel("ccp_alpha"); ax[1].set_ylabel("5-fold CV accuracy"); ax[1].set_title("CV picks the alpha")
plt.tight_layout(); plt.show()

In [ ]:
print(export_text(pruned, feature_names=list(X.columns)))
print("A tree this size is something a branch manager can actually read.")

## 6. Feature importance, done honestly 🧭

**Impurity-based** (`feature_importances_`): total weighted gain each feature produced. Free, but biased toward features with many possible thresholds (continuous `age`, `fare`) and computed on *training* data.

**Permutation importance**: shuffle one column of the **test** set, measure how much accuracy drops. Slower, unbiased, and it answers the question you actually care about: *does the model need this feature to predict well on new data?*

In [ ]:
from sklearn.inspection import permutation_importance

imp = pd.Series(pruned.feature_importances_, index=X.columns)
perm = permutation_importance(pruned, X_te, y_te, n_repeats=30, random_state=SEED)
perm_s = pd.Series(perm.importances_mean, index=X.columns)

both = pd.DataFrame({"impurity-based (train)": imp, "permutation (test)": perm_s}).sort_values("permutation (test)")
print(both.round(3))
both.plot(kind="barh"); plt.title("Two views of importance. Agree on 'sex'; disagree on the continuous features."); plt.show()

> 🧠 **Interview Q:** *"Your tree says `fare` is the 2nd most important feature. Would you tell the business that?"* — Not from impurity importance alone: `fare` is continuous (hundreds of candidate thresholds) and correlated with `pclass`, so it can steal credit. Check permutation importance, and say "fare/class" together.

## 7. 🎯 Try it yourself: pre-pruning with `min_samples_leaf` (10 min)

Keep `max_depth=None` but sweep `min_samples_leaf` over `[1, 2, 5, 10, 20, 50]`. For each print leaves, train accuracy, test accuracy.

Then answer: at which value does the train/test gap close? Is the best test score better, worse, or similar to the `ccp_alpha` result above? Which would you rather explain to a colleague?

In [ ]:
# YOUR CODE HERE
for leaf in [1, 2, 5, 10, 20, 50]:
    # t = DecisionTreeClassifier(min_samples_leaf=leaf, random_state=SEED).fit(X_tr, y_tr)
    # print leaf, t.get_n_leaves(), train acc, test acc
    pass

<details><summary>Solution</summary>

```python
for leaf in [1, 2, 5, 10, 20, 50]:
    t = DecisionTreeClassifier(min_samples_leaf=leaf, random_state=SEED).fit(X_tr, y_tr)
    print(f"min_samples_leaf={leaf:<3} leaves={t.get_n_leaves():3d}  train={t.score(X_tr, y_tr):.3f}  test={t.score(X_te, y_te):.3f}")
```
The gap closes around 10–20 samples per leaf and the test score lands in the same range as the `ccp_alpha` tree. Both are fine; `ccp_alpha` is principled (one knob, chosen by CV), `min_samples_leaf` is easier to explain ("no rule is based on fewer than 20 passengers").
</details>

## Homework starter 🏆

**Stretch task:** a Kaggle Titanic submission. Download `train.csv` and `test.csv` from https://www.kaggle.com/c/titanic/data, upload both to Colab (folder icon on the left), and run the cell. It fits a pruned tree on the same 6 features and writes `submission.csv` for upload.

> 💼 **Career note:** "Kaggle Titanic, top 30% with an interpretable pruned tree" is a fine first resume line. Beating ~0.78 needs feature engineering (title from `Name`, family size), which is Day 17's topic. Come back and improve it then.

In [ ]:
import os
if os.path.exists("train.csv") and os.path.exists("test.csv"):
    ktr, kte = pd.read_csv("train.csv"), pd.read_csv("test.csv")
    def prep(d):
        out = d[["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare"]].copy()
        out["Sex"] = (out["Sex"] == "male").astype(int)
        out["Age"] = out["Age"].fillna(ktr["Age"].median())     # trees can't take NaN in older sklearn; impute with train median
        out["Fare"] = out["Fare"].fillna(ktr["Fare"].median())
        return out
    Xk, yk = prep(ktr), ktr["Survived"]
    fullk = DecisionTreeClassifier(random_state=SEED).fit(Xk, yk)
    alphas_k = fullk.cost_complexity_pruning_path(Xk, yk).ccp_alphas[:-1]
    cvk = [cross_val_score(DecisionTreeClassifier(ccp_alpha=a, random_state=SEED), Xk, yk, cv=5).mean() for a in alphas_k]
    best = alphas_k[int(np.argmax(cvk))]
    model = DecisionTreeClassifier(ccp_alpha=best, random_state=SEED).fit(Xk, yk)
    pd.DataFrame({"PassengerId": kte["PassengerId"], "Survived": model.predict(prep(kte))}).to_csv("submission.csv", index=False)
    print(f"ccp_alpha={best:.4f}, CV accuracy={max(cvk):.3f}, leaves={model.get_n_leaves()} -> submission.csv written. Upload it on Kaggle.")
    # YOUR CODE HERE (bonus): add a 'FamilySize' = SibSp + Parch + 1 feature and re-run. Does CV improve?
else:
    print("Upload Kaggle's train.csv and test.csv to run the submission template (skipped for now).")

## Key takeaways ✅

* A **decision tree** asks one yes/no question per node, chosen greedily to maximise **information gain** = drop in **Gini** (1 − Σp²) or **entropy** (−Σ p log₂ p). Both peak at 50/50, zero at purity.
* Trees need **no scaling**, handle mixed feature types, and are **readable** (`export_text`, `plot_tree`, `decision_path`), which is why they matter in regulated industries.
* Unrestricted trees **overfit** (one passenger per leaf). Fix with **pre-pruning** (`max_depth`, `min_samples_leaf`) or **post-pruning** (`ccp_alpha` chosen from `cost_complexity_pruning_path` by CV).
* **Feature importance**: impurity-based is fast but biased toward continuous/high-cardinality features; check with **permutation importance** on held-out data.
* A single tree is **high-variance**: small data changes give a different tree. That is the problem ensembles solve.

**Tomorrow (Day 09):** bootstrap many trees, let them vote: **bagging and Random Forest**, the default first model for tabular data.